### **Extraer el `texto` y `autor` solo de las paginas con `tag = humor` y exportarlos a un archivo `JSON`**

Vamos a trabajar en el enlace:

https://quotes.toscrape.com/

#### **Crear la spider**

Dado que en el archivo **`1-quotes_to_scrape.ipynb`** realizamos todo el proceso de creación desde el Entorno virtual, instalación de Scrapy y creación del proyecto, ahora solo nos queda comenzar a crear nuestras spiders. Para ello desde la terminal ejecutamos lo siguiente:

<center><img src="https://i.postimg.cc/wMc3248y/ws-405.png"></center>

Luego, debemos editar el archivo `quotes_to_scrape_siete.py` que se crea, con el código que se muestra abajo.

#### **Paso a paso**

Vamos a inspeccionar el sitio:

<center><img src="https://i.postimg.cc/NGP2bPqQ/ws-358.png"></center>

Localizar cada `articulo` de cita:

<center><img src="https://i.postimg.cc/YqGGPfMY/ws-359.png"></center>

Localizar el `texto` de la cita:

<center><img src="https://i.postimg.cc/vZ51YY1m/ws-360.png"></center>

Localizar el `autor` de la cita:

<center><img src="https://i.postimg.cc/vmM4DJ9t/ws-361.png"></center>

Localizar los `tags` de la cita:

<center><img src="https://i.postimg.cc/tCKZGZ4f/ws-362.png"></center>

Localizar si existe un botón de `Siguiente` para pasar a la siguiente página a continuar el proceso:

<center><img src="https://i.postimg.cc/zfjVhxFZ/ws-363.png"></center>

#### **Código**

Puede proporcionar argumentos de línea de comandos a sus spiders utilizando la opción `-a` al ejecutarlos:

In [ ]:
scrapy crawl quotes -a tag=humor

Estos argumentos se pasan al método `__init__` de la Spider y se convierten en atributos de la spider por defecto.

En este ejemplo, el valor proporcionado para el argumento tag estará disponible a través de `self.tag`. Puede usar esto para hacer que su spider obtenga sólo citas con una etiqueta específica, construyendo la URL basándose en el argumento:

In [ ]:
import scrapy


class QuotesToScrapeOchoSpider(scrapy.Spider):
    name = "quotes_to_scrape_ocho"

    def start_requests(self):
        url = "https://quotes.toscrape.com/"
        tag = getattr(self, "tag", None)
        if tag is not None:
            url = url + "tag/" + tag
        yield scrapy.Request(url, self.parse)

    def parse(self, response):
        # for quote in response.xpath('//div[@class="quote"]'):
        for quote in response.css("div.quote"):
            yield {
                # 'text': quote.xpath('./span[@class="text"]/text()').get()
                "text": quote.css("span.text::text").get(),
                # 'author': quote.xpath('.//small[@class="author"]/text()').get()
                "author": quote.css("small.author::text").get() 
            } 

        # next_page = response.xpath('//li[@class="next"]/a/@href').get()     
        next_page = response.css("li.next a::attr(href)").get()
        if next_page is not None:
            yield response.follow(next_page, self.parse)

Si pasa el argumento `tag=humor` a esta spider, observará que sólo visitará las URL de la etiqueta `humor`, como `https://quotes.toscrape.com/tag/humor`.

#### **Test metodo `getattr()`**

In [ ]:
class Person:
  name = "John"
  age = 36
  country = "Norway"

x = getattr(Person, 'age')
x

In [ ]:
class Person:
  name = "John"
  age = 36
  country = "Norway"

alfonso = Person()

x = getattr(alfonso, 'age', 38)
x

36

In [ ]:
class Person:
  name = "John"
  age = 36
  country = "Norway"

  def __init__(self, tag):
    self.tag = tag
  
  def test(self):
    return getattr(self, "tag", None)

alfonso = Person('humor')
alfonso.test()


'humor'

#### **Ejecución**

Nos posicionamos en la ruta correcta y ejecutamos nuestra Spider:

In [ ]:
scrapy crawl quotes_to_scrape_ocho -a tag=humor

<center><img src="https://i.postimg.cc/9XG4c0Rm/ws-406.png"></center>

De esta manera obtenemos toda la info que pedimos extraer del website:

<center><img src="https://i.postimg.cc/0yNrGCtj/ws-407.png"></center>

Podriamos exportar toda esta data en un archivo `JSON` escribiendo el siguiente comando:

In [ ]:
scrapy crawl quotes_to_scrape_ocho -o quotes_to_scrape_ocho.json -a tag=humor

<center><img src="https://i.postimg.cc/QxhFCD1p/ws-408.png"></center>